# HDB and School Feature Engineering

This notebook combines cleaned HDB resale data with school quality indicators to create distance-based school accessibility features for the final modelling dataset.

## Cleaning objective
- load cleaned HDB, MRT, and school quality datasets
- compute distance from each HDB transaction to the CBD
- geocode school locations with OneMap
- create yearly counts and indicators for nearby schools
- export the final feature table to the project `data` folder


In [ ]:
# ============================================================
# 1. Imports and configuration
# ============================================================

import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import haversine_distances
from sklearn.neighbors import BallTree

MRT_FILE = Path("../../data/cleaned/mrt_cleaned.csv")
HDB_FILE = Path("../../data/cleaned/hdb_final.csv")
SCHOOL_FILE = Path("../../outputs/Good_School_index.csv")
OUTPUT_FILE = Path("../../data/final_df.csv")

SEARCH_URL = "https://www.onemap.gov.sg/api/common/elastic/search"
EARTH_RADIUS_KM = 6371
GOOD_SCHOOL_COLS = ["good_school_75", "good_school_80", "good_school_85", "good_school_90"]


## Load source datasets and derive CBD distance

This section loads the cleaned inputs, identifies Raffles Place as the CBD reference point, and computes the HDB-to-CBD distance used later in the modelling dataset.


In [ ]:
# ============================================================
# 2. Load data and compute CBD distance
# ============================================================

mrt = pd.read_csv(MRT_FILE)
hdb_final = pd.read_csv(HDB_FILE)
sch_data = pd.read_csv(SCHOOL_FILE)

cbd_row = mrt.loc[mrt["station_name"].str.upper() == "RAFFLES PLACE"]
if cbd_row.empty:
    raise ValueError("Could not find `RAFFLES PLACE` in the MRT reference file.")

cbd_lat = cbd_row["lat"].iloc[0]
cbd_lon = cbd_row["lon"].iloc[0]

hdb_rad = np.radians(hdb_final[["lat", "lon"]].values)
cbd_rad = np.radians([[cbd_lat, cbd_lon]])
dist_to_cbd = haversine_distances(hdb_rad, cbd_rad)

hdb_final["dist_to_cbd_km"] = dist_to_cbd.flatten() * EARTH_RADIUS_KM

print("HDB records:", len(hdb_final))
print("School records:", len(sch_data))
hdb_final[["lat", "lon", "dist_to_cbd_km"]].head()


## Geocode school locations

The helper functions below normalize school names, apply a small set of manual overrides, and query OneMap to recover latitude and longitude coordinates for each school.


In [ ]:
# ============================================================
# 3. Geocoding helper functions
# ============================================================

def geocode_onemap(
    search_val: str,
    session: requests.Session | None = None,
    auth_token: str | None = None,
    retries: int = 3,
    sleep: float = 0.3,
) -> tuple[float | None, float | None]:
    """Geocode a location string with OneMap and return latitude and longitude."""
    request_session = session or requests.Session()
    headers = {"Authorization": auth_token} if auth_token else {}
    params = {
        "searchVal": search_val,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1,
    }

    for attempt in range(retries):
        try:
            response = request_session.get(
                SEARCH_URL,
                params=params,
                headers=headers,
                timeout=15,
            )
            response.raise_for_status()
            payload = response.json()
            if int(payload.get("found", 0)) > 0:
                best_match = payload["results"][0]
                return float(best_match["LATITUDE"]), float(best_match["LONGITUDE"])
            return None, None
        except requests.RequestException:
            if attempt == retries - 1:
                return None, None
            time.sleep((attempt + 1) * sleep)

    return None, None


def normalize_name(name: str) -> str:
    """Normalize school names before geocoding."""
    if pd.isna(name):
        return name

    normalized = str(name).strip()
    normalized = unicodedata.normalize("NFKC", normalized)
    normalized = normalized.replace("’", "'").replace("‘", "'")
    return normalized


school_overrides = {
    "CHIJ (Toa Payoh)": "CHIJ Primary (Toa Payoh)",
    "St. Andrew's Junior": "St. Andrew's Junior School",
    "St. Anthony's": "St. Anthony's Primary School",
    "St. Anthony's Canossian": "St. Anthony's Canossian Primary School",
    "St. Gabriel's": "St. Gabriel's Primary School",
    "St. Hilda's": "St. Hilda's Primary School",
    "St. Joseph's Institution Junior": "St. Joseph's Institution Junior",
    "St. Margaret's": "St. Margaret's Primary School",
    "St. Stephen's": "St. Stephen's School",
    "CHIJ (Katong)": "CHIJ Katong Primary",
    "CHIJ Our Lady of Good Counsel": "CHIJ Our Lady of Good Counsel",
    "CHIJ Our Lady Queen of Peace": "CHIJ Our Lady Queen of Peace",
    "CHIJ (Kellock)": "CHIJ Kellock",
}


def geocode_school(name: str, session: requests.Session | None = None) -> tuple[float | None, float | None, str | None]:
    """Geocode a school name using manual overrides and fallback queries."""
    queries = [school_overrides[name]] if name in school_overrides else [
        f"{name} Primary School",
        f"{name} School",
    ]

    for query in queries:
        lat, lon = geocode_onemap(query, session=session)
        if lat is not None and lon is not None:
            return lat, lon, query

    return None, None, None


In [ ]:
# ============================================================
# 4. Geocode school dataset
# ============================================================

session = requests.Session()

sch_data["search_school"] = sch_data["school"].map(normalize_name)
unique_schools = sch_data["search_school"].dropna().unique()

school_coordinate_map = {}
school_query_map = {}

for school_name in unique_schools:
    lat, lon, query_used = geocode_school(school_name, session=session)
    school_coordinate_map[school_name] = (lat, lon)
    school_query_map[school_name] = query_used

sch_data["lat"] = sch_data["search_school"].map(lambda value: school_coordinate_map.get(value, (None, None))[0])
sch_data["lon"] = sch_data["search_school"].map(lambda value: school_coordinate_map.get(value, (None, None))[1])
sch_data["query_used"] = sch_data["search_school"].map(school_query_map)

missing_school_coords = sch_data.loc[
    sch_data["lat"].isna() | sch_data["lon"].isna(),
    ["school", "search_school", "lat", "lon"],
]
print("Schools with missing lat/lon:", len(missing_school_coords))
missing_school_coords.head()


## Create yearly school-accessibility features

This section matches HDB transactions to same-year school records and generates counts and binary indicators for schools within 1 km and 1-2 km bands.


In [ ]:
# ============================================================
# 5. Build school proximity features
# ============================================================

merged_df = hdb_final.copy()
sch_df = sch_data.copy()

merged_df["year"] = pd.to_datetime(merged_df["month"]).dt.year
sch_df["year"] = sch_df["year"].astype(int)

merged_df = merged_df.dropna(subset=["lat", "lon"]).copy()
sch_df = sch_df.dropna(subset=["lat", "lon"]).copy()

for column in GOOD_SCHOOL_COLS:
    sch_df[column] = sch_df[column].fillna(0).astype(int)

merged_df["Countall_<1"] = 0
merged_df["Countall_1-2"] = 0

for column in GOOD_SCHOOL_COLS:
    merged_df[f"Count{column}_<1"] = 0
    merged_df[f"Count{column}_1-2"] = 0
    merged_df[f"D_1_{column}"] = 0
    merged_df[f"D_1-2_{column}"] = 0

for year in sorted(merged_df["year"].dropna().unique()):
    house_index = merged_df.index[merged_df["year"] == year]
    school_year = sch_df.loc[sch_df["year"] == year].copy()

    if school_year.empty:
        continue

    house_year = merged_df.loc[house_index]
    house_coords = np.radians(house_year[["lat", "lon"]].to_numpy())
    school_coords = np.radians(school_year[["lat", "lon"]].to_numpy())
    tree = BallTree(school_coords, metric="haversine")

    neighbours_1km = tree.query_radius(house_coords, r=1 / EARTH_RADIUS_KM)
    neighbours_2km = tree.query_radius(house_coords, r=2 / EARTH_RADIUS_KM)

    countall_1km = np.array([len(indexes) for indexes in neighbours_1km])
    countall_2km = np.array([len(indexes) for indexes in neighbours_2km])

    merged_df.loc[house_index, "Countall_<1"] = countall_1km
    merged_df.loc[house_index, "Countall_1-2"] = countall_2km - countall_1km

    for column in GOOD_SCHOOL_COLS:
        school_values = school_year[column].to_numpy()
        countgood_1km = np.array([school_values[indexes].sum() for indexes in neighbours_1km])
        countgood_2km = np.array([school_values[indexes].sum() for indexes in neighbours_2km])

        merged_df.loc[house_index, f"Count{column}_<1"] = countgood_1km
        merged_df.loc[house_index, f"Count{column}_1-2"] = countgood_2km - countgood_1km
        merged_df.loc[house_index, f"D_1_{column}"] = (countgood_1km > 0).astype(int)
        merged_df.loc[house_index, f"D_1-2_{column}"] = ((countgood_2km - countgood_1km) > 0).astype(int)

merged_df = merged_df.loc[merged_df["year"] >= 2013].copy()
print("Final records after dropping pre-2013 rows:", len(merged_df))
merged_df.head()


In [ ]:
# ============================================================
# 6. Rename columns and save output
# ============================================================

merged_df = merged_df.rename(
    columns={
        "Countall_<1": "countall_0_1km",
        "Countall_1-2": "countall_1_2km",
        "Countgood_school_75_<1": "count_0_1km_good75",
        "Countgood_school_75_1-2": "count_1_2km_good75",
        "Countgood_school_80_<1": "count_0_1km_good80",
        "Countgood_school_80_1-2": "count_1_2km_good80",
        "Countgood_school_85_<1": "count_0_1km_good85",
        "Countgood_school_85_1-2": "count_1_2km_good85",
        "Countgood_school_90_<1": "count_0_1km_good90",
        "Countgood_school_90_1-2": "count_1_2km_good90",
        "D_1_good_school_75": "d_0_1km_good75",
        "D_1-2_good_school_75": "d_1_2km_good75",
        "D_1_good_school_80": "d_0_1km_good80",
        "D_1-2_good_school_80": "d_1_2km_good80",
        "D_1_good_school_85": "d_0_1km_good85",
        "D_1-2_good_school_85": "d_1_2km_good85",
        "D_1_good_school_90": "d_0_1km_good90",
        "D_1-2_good_school_90": "d_1_2km_good90",
    }
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved final feature dataset to {OUTPUT_FILE.resolve()}")
merged_df.head()
